# CardioSafe AI - Notebook 05: explainability with a faithfulness test

We explain one trained model (**xResNet1D, seed 0**, the best architecture on average) with two methods:
- **Grad-CAM:** which *time regions* drove the prediction (coarse, fast).
- **Integrated Gradients (IG):** which *lead and time point* drove it (fine-grained, with a built-in correctness check, the completeness error).

A heatmap that looks plausible is not evidence that it is faithful. So we test it: **mask the regions the method calls most important and see whether the model's confidence really drops**, compared with masking random regions (should drop little) and the *least* important regions (should drop least).

**Runtime: GPU.** About 5 minutes. Needs notebooks 02 and 03 outputs (`models/xresnet1d_s0.pt`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
WORK = '/content/drive/MyDrive/CardioSafe'
CKPT = WORK + '/models'
sys.path.insert(0, WORK)

In [ ]:
%%writefile /content/drive/MyDrive/CardioSafe/cardiosafe_explain.py
"""Explainability for 1D ECG models: Grad-CAM, Integrated Gradients, and a deletion-based faithfulness test."""
import numpy as np
import torch
import torch.nn.functional as Fn


def gradcam(model, layer, x, cls, batch=64):
    """Grad-CAM over time. `layer` = conv feature block (e.g. model.body[5]). Returns (N, T) in [0, 1]."""
    model.eval()
    T = x.shape[-1]
    out = []
    for i in range(0, len(x), batch):
        xb = x[i:i + batch]
        store = {}
        h = layer.register_forward_hook(lambda m, inp, o: store.__setitem__('a', o))
        logits = model(xb)
        h.remove()
        a = store['a']
        g = torch.autograd.grad(logits[:, cls].sum(), a)[0]
        w = g.mean(-1, keepdim=True)
        cam = torch.relu((w * a).sum(1, keepdim=True))
        cam = Fn.interpolate(cam, size=T, mode='linear', align_corners=False)[:, 0]
        cam = cam / (cam.amax(1, keepdim=True) + 1e-8)
        out.append(cam.detach().cpu())
    return torch.cat(out).numpy()


def integrated_gradients(model, x, cls, steps=32, batch=32):
    """IG with zero baseline (the training-set mean signal, since inputs are standardised) and midpoint Riemann sum.
    Returns attributions (N, 12, T) and the completeness error: |sum(attr) - (f(x) - f(0))| / |f(x) - f(0)|."""
    model.eval()
    attrs, errs = [], []
    zero = torch.zeros_like(x[:1])
    with torch.no_grad():
        f0 = model(zero)[0, cls]
    for i in range(0, len(x), batch):
        xb = x[i:i + batch]
        total = torch.zeros_like(xb)
        for k in range(steps):
            xi = ((k + 0.5) / steps * xb).clone().requires_grad_(True)
            g = torch.autograd.grad(model(xi)[:, cls].sum(), xi)[0]
            total += g
        at = (xb * total / steps).detach()
        with torch.no_grad():
            diff = model(xb)[:, cls] - f0
        errs.append(((at.sum((1, 2)) - diff).abs() / (diff.abs() + 1e-6)).cpu())
        attrs.append(at.cpu())
    return torch.cat(attrs).numpy(), torch.cat(errs).numpy()


def window_scores(imp, w=10):
    """(N, T) importance -> (N, T // w) mean importance per window."""
    N, T = imp.shape
    return imp[:, :T // w * w].reshape(N, T // w, w).mean(-1)


@torch.no_grad()
def deletion_curve(model, x, cls, scores, fracs, mode='top', w=10, seed=0, bs=256):
    """Mask (set to the baseline 0) the `frac` most ('top'), least ('bottom') or randomly chosen windows,
    return the mean predicted probability of class `cls` at each frac. A faithful explanation makes 'top' fall fastest."""
    model.eval()
    N, nW = scores.shape
    T = x.shape[-1]
    if mode == 'random':
        scores = np.random.default_rng(seed).random(scores.shape)
    order = np.argsort(-scores if mode != 'bottom' else scores, axis=1)
    rank = np.argsort(order, axis=1)
    res = []
    for f in fracs:
        k = int(round(f * nW))
        win = torch.from_numpy(rank < k)                           # (N, nW) True = masked
        full = win.repeat_interleave(w, dim=1)                     # (N, nW*w)
        if full.shape[1] < T:
            full = torch.cat([full, torch.zeros(N, T - full.shape[1], dtype=torch.bool)], dim=1)
        keep = (~full).float().to(x.device)[:, None, :]
        ps = [torch.sigmoid(model(x[i:i + bs] * keep[i:i + bs])[:, cls]).float().cpu() for i in range(0, N, bs)]
        res.append(torch.cat(ps).mean().item())
    return np.array(res)


In [ ]:
import importlib, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import cardiosafe_common as C, cardiosafe_models as MZ, cardiosafe_explain as E
importlib.reload(E)

X, Y, fold, tr, va, te, meta = C.load_data(WORK + '/ptbxl_100hz_super5.npz')
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mu, sd = C.standardize_fit(X[tr])
Xte = C.to_tensor_ncl(X[te], mu, sd, dev); yt = Y[te]

net = MZ.REGISTRY['xresnet1d']().to(dev)
net.load_state_dict(torch.load(f'{CKPT}/xresnet1d_s0.pt', map_location=dev)); net.eval()
pt = C.predict(net, Xte)
print('Loaded xresnet1d_s0 | test macro AUROC:', round(C.macro_auc(yt, pt), 4), '(notebook 03 reported 0.9241)')
LAYER = net.body[5]          # end of stage 3 (about 63 time steps), a good resolution/semantics trade-off

## 1. Explain true-positive predictions
For each class we take up to 200 test ECGs that truly have the finding and that the model detects (probability > 0.5).

In [ ]:
rng = np.random.default_rng(0)
expl = {}
for c, n in enumerate(C.CLASSES):
    idx = np.where((yt[:, c] == 1) & (pt[:, c] > 0.5))[0]
    if len(idx) < 20:                                   # safety fallback
        idx = np.argsort(-(pt[:, c] * (yt[:, c] == 1)))[:20]
    rng.shuffle(idx); idx = np.sort(idx[:200])
    xs = Xte[torch.as_tensor(idx, device=dev)]
    cam = E.gradcam(net, LAYER, xs, c)
    ig, err = E.integrated_gradients(net, xs, c, steps=32)
    expl[c] = dict(idx=idx, xs=xs, cam=cam, ig=ig, p=pt[idx, c])
    print(f'{n:5s} n={len(idx):3d}  IG completeness error: median {np.median(err):.3f}, 90th pct {np.quantile(err, .9):.3f}')

## 2. Faithfulness test (deletion)
Mask the top-k% windows (0.1 s each) of the signal, by each method's importance ranking, and measure the model's probability for the finding. Lower probability after masking = the method identified regions the model really relies on.
Note: masking replaces the signal with the training-mean baseline, which is itself unnatural input, so we compare against random and least-important masking rather than reading absolute values.

In [ ]:
fracs = [0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
curves = {k: [] for k in ['GradCAM (top)', 'IG+ (top)', 'IG |abs| (top)', 'random', 'GradCAM (bottom)', 'IG+ (bottom)']}
per_class = []
for c, n in enumerate(C.CLASSES):
    d = expl[c]
    sc = {'GradCAM': E.window_scores(d['cam']),
          'IG+': E.window_scores(np.clip(d['ig'], 0, None).sum(1)),
          'IG |abs|': E.window_scores(np.abs(d['ig']).sum(1))}
    cv = {'GradCAM (top)': E.deletion_curve(net, d['xs'], c, sc['GradCAM'], fracs, 'top'),
          'IG+ (top)': E.deletion_curve(net, d['xs'], c, sc['IG+'], fracs, 'top'),
          'IG |abs| (top)': E.deletion_curve(net, d['xs'], c, sc['IG |abs|'], fracs, 'top'),
          'random': np.mean([E.deletion_curve(net, d['xs'], c, sc['IG+'], fracs, 'random', seed=s) for s in range(5)], 0),
          'GradCAM (bottom)': E.deletion_curve(net, d['xs'], c, sc['GradCAM'], fracs, 'bottom'),
          'IG+ (bottom)': E.deletion_curve(net, d['xs'], c, sc['IG+'], fracs, 'bottom')}
    for k in curves: curves[k].append(cv[k])
    for k, v in cv.items():
        per_class.append({'class': n, 'method': k, **{f'p@{int(f * 100)}%': v[i] for i, f in enumerate(fracs) if f in (0.1, 0.2, 0.3)},
                          'deletion_AUC': (np.trapezoid if hasattr(np, 'trapezoid') else np.trapz)(v, fracs)})
pc = pd.DataFrame(per_class)
pc.to_csv(WORK + '/results_05_faithfulness_per_class.csv', index=False)
print('Mean probability of the finding after masking the top-k% windows (rows = method, averaged over classes):')
mean_curves = pd.DataFrame({k: np.mean(v, 0) for k, v in curves.items()}, index=[f'{int(f * 100)}%' for f in fracs]).T
display(mean_curves.round(3))
mean_curves.to_csv(WORK + '/results_05_faithfulness.csv')
print('Deletion AUC per class (lower = more faithful; compare each method with random):')
display(pc.pivot(index='method', columns='class', values='deletion_AUC').round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
sty = {'GradCAM (top)': ('#d62728', '-'), 'IG+ (top)': ('#1f77b4', '-'), 'IG |abs| (top)': ('#2ca02c', '-'),
       'random': ('k', '--'), 'GradCAM (bottom)': ('#d62728', ':'), 'IG+ (bottom)': ('#1f77b4', ':')}
for k, (col, ls) in sty.items():
    ax.plot([f * 100 for f in fracs], np.mean(curves[k], 0), ls, color=col, marker='o', ms=4, label=k)
ax.set_xlabel('% of the signal masked (windows ranked by importance)'); ax.set_ylabel('mean predicted probability of the true finding')
ax.set_title('Faithfulness: deletion test'); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(WORK + '/fig_faithfulness.png', dpi=150); plt.show()

## 3. What the explanations look like

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(12, 13))
for c, n in enumerate(C.CLASSES):
    d = expl[c]; j = int(np.argmax(d['p']))
    sig = d['xs'][j].cpu().numpy(); lead = int(np.abs(d['ig'][j]).sum(1).argmax())
    ax = axes[c]; t = np.arange(sig.shape[1]) / 100
    ax.imshow(d['cam'][j][None, :], aspect='auto', cmap='Reds', alpha=.55, extent=[0, t[-1], sig[lead].min() - .5, sig[lead].max() + .5], vmin=0, vmax=1)
    ax.plot(t, sig[lead], 'k', lw=.9)
    ax.set_title(f'{n}: model probability {d["p"][j]:.2f}; most influential lead = {C.LEADS[lead]}; red shading = Grad-CAM importance', fontsize=10)
    ax.set_xlabel('time (s)' if c == 4 else ''); ax.set_ylabel('standardised amplitude')
plt.tight_layout(); plt.savefig(WORK + '/fig_gradcam_examples.png', dpi=150); plt.show()

In [ ]:
# Integrated Gradients across all 12 leads for the highest-confidence MI example
c = 1; d = expl[c]; j = int(np.argmax(d['p']))
sig = d['xs'][j].cpu().numpy(); att = np.clip(d['ig'][j], 0, None)
fig, axes = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={'height_ratios': [1, 1.1]})
off = np.arange(12)[:, None] * 6
axes[0].plot(np.arange(sig.shape[1]) / 100, (sig + off).T, lw=.7); axes[0].set_yticks(off[:, 0]); axes[0].set_yticklabels(C.LEADS)
axes[0].set_title(f'MI example (model probability {d["p"][j]:.2f}): 12-lead signal'); 
im = axes[1].imshow(att, aspect='auto', cmap='magma', extent=[0, sig.shape[1] / 100, 12, 0])
axes[1].set_yticks(np.arange(12) + .5); axes[1].set_yticklabels(C.LEADS); axes[1].set_xlabel('time (s)')
axes[1].set_title('Integrated Gradients (positive evidence for MI), by lead and time'); plt.colorbar(im, ax=axes[1], pad=.01)
plt.tight_layout(); plt.savefig(WORK + '/fig_ig_mi_example.png', dpi=150); plt.show()

In [ ]:
# Which leads does the model rely on, per class? (plausibility check, NOT clinical validation)
M_ = np.stack([np.abs(expl[c]['ig']).sum(2).mean(0) for c in range(5)]); M_ = M_ / M_.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(10, 3.6))
im = ax.imshow(M_, cmap='viridis', aspect='auto'); ax.set_xticks(range(12)); ax.set_xticklabels(C.LEADS); ax.set_yticks(range(5)); ax.set_yticklabels(C.CLASSES)
for i in range(5):
    for k in range(12): ax.text(k, i, f'{M_[i, k]:.2f}', ha='center', va='center', color='w', fontsize=8)
ax.set_title('Share of Integrated-Gradients attribution per lead'); plt.colorbar(im, ax=ax); plt.tight_layout()
plt.savefig(WORK + '/fig_lead_importance.png', dpi=150); plt.show()
print('Top 3 leads per class:', {C.CLASSES[i]: [C.LEADS[k] for k in np.argsort(-M_[i])[:3]] for i in range(5)})

## Next
Send me: the mean-curves table, the per-class deletion AUC table, the IG completeness-error lines, and (as pictures) `fig_faithfulness.png`, `fig_gradcam_examples.png` and `fig_lead_importance.png`.
Then notebook 06: QTc measurement and the combined cardiac-safety knowledge base.